In [1]:
import pandas as pd
from langchain.schema import Document

In [2]:
# Charger le CSV
csv_path = "../../RGBD/table_produits/produits.csv"   # Remplace avec le chemin réel si besoin
df = pd.read_csv(csv_path, encoding='utf-8')

# Vérifier que la colonne "description" existe bien
if "description" not in df.columns:
    raise ValueError("La colonne 'description' n'existe pas dans le CSV ! Vérifie le nom des colonnes.")
# Vérification de la colonne "id_produit"
if "id_produit" not in df.columns:
    raise ValueError("La colonne 'id_produit' n'existe pas dans le CSV ! Vérifiez le nom des colonnes.")

# Transformer chaque ligne en objet Document avec métadonnées
documents = []
for index, row in df.iterrows():
    metadata = {"source": csv_path, "row": index, "id_produit": row["id_produit"]}

    # Ajouter les métadonnées supplémentaires si elles existent dans le DataFrame
    if "saveur" in row:
        metadata["saveur"] = str(row["saveur"])
    if "pg_vg" in row:
        metadata["pg_vg"] = str(row["pg_vg"])
    if "origine" in row:
        metadata["origine"] = str(row["origine"])
    if "frais" in row:
        metadata["frais"] = str(row["frais"])
    if "surbooste" in row:
        metadata["surbooste"] = str(row["surbooste"])
    if "brand" in row:
        metadata["brand"] = str(row["brand"])
    if "gout" in row:
        metadata["gout"] = str(row["gout"])

    documents.append(Document(page_content=str(row["description"]), metadata=metadata))

print(f"Nombre de documents préparés : {len(documents)}")


Nombre de documents préparés : 1531


In [3]:
documents[:5]

[Document(metadata={'source': '../../RGBD/table_produits/produits.csv', 'row': 0, 'id_produit': 1, 'saveur': 'cassis, lime, fruits rouges, parfait', 'pg_vg': '50/50', 'origine': 'France', 'frais': 'Oui', 'surbooste': 'Non', 'brand': 'Arômes et Liquides (A&L)', 'gout': 'fruit'}, page_content="Le e-liquide Ragnarok par A&L Ultimate est un produitprêt à l'emploipour cigarette électronique. Dans sa fiole 10ml, vous retrouverez des saveurs de fruits rouges, accompagnées d'un goût frais de cassis. Idéal pour débuter, ce e-liquide vous est proposé en différents taux de nicotine au choix : 0, 3, 6, ou 12mg/ml. Qu'est-ce que le Ragnarok Ultimate ? Le Ragnarok, c'est unerecette fraiche aux goûts de fruitsdehaute qualité française, signée Arômes et Liquides (A&L). Ici, vous en profiterez dans sa version e-liquide prêt à vaper. Conforme à la loi TPD, le e liquide Ragnarok est présenté en fiole PET de 10ml. Pourquoi ? Afin de vous assurer unproduit déjà nicotiné à votre convenance! Pourquoi choisir

In [4]:
lengths = [len(doc.page_content) for doc in documents]
print(f"Taille moyenne des documents : {sum(lengths) / len(lengths)} caractères")

Taille moyenne des documents : 3119.184846505552 caractères


1. Build a synthetic dataset for evaluation
Tout d'abord, nous construisons un ensemble de données synthétique composé de questions et de contextes associés. La méthode consiste à extraire des éléments de notre base de connaissances et à demander à un modèle de langue (LLM) de générer des questions basées sur ces documents.

Ensuite, nous mettons en place d'autres agents de LLM pour agir en tant que filtres de qualité pour les couples question-réponse générés : chacun d'eux agira comme le filtre pour un défaut spécifique.










1.1. Prepare source documents


In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialiser le découpeur de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # Taille des segments
    chunk_overlap=100,  # Chevauchement entre segments
    add_start_index=True,  # Ajouter un index de début
    separators=["\n\n", "\n", ".", " ", ""],  # Séparateurs de texte
)

In [6]:
# Découper les documents
docs_processed = text_splitter.split_documents(documents)

# Afficher un aperçu des segments découpés
for doc in docs_processed[:5]:  # Afficher les 5 premiers segments
    print(doc.page_content)
    print("-" * 50)

Le e-liquide Ragnarok par A&L Ultimate est un produitprêt à l'emploipour cigarette électronique. Dans sa fiole 10ml, vous retrouverez des saveurs de fruits rouges, accompagnées d'un goût frais de cassis. Idéal pour débuter, ce e-liquide vous est proposé en différents taux de nicotine au choix : 0, 3, 6, ou 12mg/ml. Qu'est-ce que le Ragnarok Ultimate ? Le Ragnarok, c'est unerecette fraiche aux goûts de fruitsdehaute qualité française, signée Arômes et Liquides (A&L). Ici, vous en profiterez dans sa version e-liquide prêt à vaper. Conforme à la loi TPD, le e liquide Ragnarok est présenté en fiole PET de 10ml. Pourquoi ? Afin de vous assurer unproduit déjà nicotiné à votre convenance! Pourquoi choisir le Ragnarok en e-liquide 10ml ? Simple, pratique et efficace, le e-liquide Ragnarok en 10ml permet de démarrer sans attendre votre séance de vapotage. Déjà minutieusement travaillé, dosé et nicotiné, il vous suffit de le verser dans votre réservoir de cigarette électronique pour l'utiliser !

In [7]:
print(f"Nombre de documents avant découpage : {len(documents)}")
print(f"Nombre de documents après découpage : {len(docs_processed)}")

Nombre de documents avant découpage : 1531
Nombre de documents après découpage : 3097


In [9]:
unique_products = {}
filtered_docs_processed = []

for doc in docs_processed:
    id_produit = doc.metadata['id_produit']
    if id_produit not in unique_products:
        unique_products[id_produit] = True
        filtered_docs_processed.append(doc)

filtered_docs_processed[:5]

[Document(metadata={'source': '../../RGBD/table_produits/produits.csv', 'row': 0, 'id_produit': 1, 'saveur': 'cassis, lime, fruits rouges, parfait', 'pg_vg': '50/50', 'origine': 'France', 'frais': 'Oui', 'surbooste': 'Non', 'brand': 'Arômes et Liquides (A&L)', 'gout': 'fruit', 'start_index': 0}, page_content="Le e-liquide Ragnarok par A&L Ultimate est un produitprêt à l'emploipour cigarette électronique. Dans sa fiole 10ml, vous retrouverez des saveurs de fruits rouges, accompagnées d'un goût frais de cassis. Idéal pour débuter, ce e-liquide vous est proposé en différents taux de nicotine au choix : 0, 3, 6, ou 12mg/ml. Qu'est-ce que le Ragnarok Ultimate ? Le Ragnarok, c'est unerecette fraiche aux goûts de fruitsdehaute qualité française, signée Arômes et Liquides (A&L). Ici, vous en profiterez dans sa version e-liquide prêt à vaper. Conforme à la loi TPD, le e liquide Ragnarok est présenté en fiole PET de 10ml. Pourquoi ? Afin de vous assurer unproduit déjà nicotiné à votre convenance

In [10]:
filtered_docs_processed = filtered_docs_processed[:50]  # Limiter à 50 documents
print(f"Nombre de documents après découpage : {len(filtered_docs_processed)}")

Nombre de documents après découpage : 50


1.2. Setup agents for question generation
We use Mixtral for QA couple generation because it it has excellent performance in leaderboards such as Chatbot Arena.

In [11]:
from huggingface_hub import InferenceClient
import json

repo_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"

llm_client = InferenceClient(
    model=repo_id,
    timeout=120,
)


def call_llm(inference_client: InferenceClient, prompt: str):
    response = inference_client.post(
        json={
            "inputs": prompt,
            "parameters": {"max_new_tokens": 1000},
            "task": "text-generation",
        },
    )
    return json.loads(response.decode())[0]["generated_text"]


call_llm(llm_client, "This is a test context")

c:\Users\antoa\anaconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'This is a test context for the `@mui/material` library.\n\n## Installation\n\n```sh\nnpm install @mui/material\n```\n\n## Usage\n\n```jsx\nimport React from \'react\';\nimport { Button } from \'@mui/material\';\n\nfunction App() {\n  return (\n    <div className="App">\n      <Button variant="contained" color="primary">\n        Hello World\n      </Button>\n    </div>\n  );\n}\n\nexport default App;\n```\n\n## Documentation\n\n- [Material-UI](https://material-ui.com/)\n- [Material Design](https://material.io/)'

In [12]:
QA_generation_prompt = """
Votre tâche est de rédiger une question factuelle et une réponse à partir d'un contexte concernant un produit e-liquide.
Votre question factuelle doit pouvoir être répondue par un renseignement factuel précis et concis tiré du contexte.
Votre question factuelle doit être formulée de la même manière que les questions qu'un utilisateur pourrait poser dans un moteur de recherche.
Cela signifie que votre question factuelle NE DOIT PAS mentionner des termes comme "selon le passage" ou "contexte".

Veuillez fournir votre réponse de la manière suivante :

Sortie:::
Question factuelle : (votre question factuelle sur le produit e-liquide)
Réponse : (votre réponse à la question factuelle)

Voici maintenant le contexte.

Contexte : {context}\n
Sortie:::"""


# Génération de Couples Question-Réponse (QA)

Dans ce projet, nous générons des couples question-réponse (QA) à partir de notre base de connaissances. Normalement, pour garantir une bonne couverture et une diversité suffisante, il faudrait générer au moins **200** tests simples.  

Cependant, étant donné que nous utilisons **Hugging Face** et que nous sommes limités en ressources, nous réduisons ce nombre à **20** couples QA. Cette sélection plus restreinte nous permet d'optimiser l'utilisation des ressources tout en maintenant une qualité acceptable pour l'évaluation.  

Nous nous assurerons que ces 20 couples couvrent un éventail représentatif des cas possibles afin de maximiser leur utilité dans notre processus de validation.


In [13]:
import random
from tqdm import tqdm

N_GENERATIONS = 50  # We intentionally generate only 10 QA couples here for cost and time considerations

print(f"Generating {N_GENERATIONS} QA couples...")
outputs = []
MAX_ANSWER_LENGTH = 300  # Limite de la longueur de la réponse

for sampled_context in tqdm(random.sample(docs_processed, N_GENERATIONS)):
    # Génération d'un couple QA
    output_QA_couple = call_llm(llm_client, QA_generation_prompt.format(context=sampled_context.page_content))
    try:
        question = output_QA_couple.split("Factoid question: ")[-1].split("Answer: ")[0]
        answer = output_QA_couple.split("Answer: ")[-1]
        
        # Limite la longueur de la réponse à 300 caractères
        if len(answer) > MAX_ANSWER_LENGTH:
            answer = answer[:MAX_ANSWER_LENGTH]  # Tronque la réponse si elle est trop longue
        
        outputs.append(
            {
                "context": sampled_context.page_content,
                "question": question,
                "answer": answer,
                "source_doc": sampled_context.metadata["source"],
            }
        )
    except Exception as e:
        print(f"Error processing context: {sampled_context.page_content}")
        print(f"Error: {e}")
        continue  # Continue si une erreur survient

# Vérifiez si 'outputs' contient plusieurs éléments avant de créer le DataFrame
if outputs:
    # Création du DataFrame
    df = pd.DataFrame(outputs)
    print(f"Dataframe created with {len(df)} rows.")
else:
    print("No valid QA couples were generated.")


Generating 50 QA couples...


100%|██████████| 50/50 [00:45<00:00,  1.10it/s]

Dataframe created with 50 rows.


In [14]:
df.to_csv('QA_test_samples.csv', index=False)

In [31]:
df_sample = pd.read_csv('QA_test_samples.csv')

In [32]:
df_sample

,context,question,answer,source_doc
0,La marque française Secret's Lab complète sa c...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
1,Le Mr Happy Pie de chez Jin & Juice vous propo...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
2,. Astuce : pensez à adapter votre matériel au ...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
3,"Issu de la gamme WSalt Flavors de Liquideo, le...",\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
4,. Astuce : pensez à adapter votre matériel au ...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
5,Le Cocktail 100ml est un e-liquide à booster i...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
6,. Si vous utilisez une cigarette électronique ...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
7,. Quel taux de nicotine choisir pour votre e-l...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
8,. Astuce : pensez à adapter votre matériel au ...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv
9,. Nos conseils pour la maturation de votre e-l...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv


In [19]:
import re

In [33]:
# Fonction pour extraire le texte entre la deuxième occurrence de "Question factuelle :" et "Réponse :"
def extract_second_q_and_r(text):
    # Utilisation d'une expression régulière pour capturer le texte entre la deuxième Question factuelle et Réponse
    pattern = r"(?:Question factuelle :.*?)(Question factuelle :.*?)(Réponse :.*?)(?=Question factuelle :|$)"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).replace('Question factuelle :', '').replace('Réponse :', '').strip()
    else:
        return None

# Appliquer la fonction pour créer une nouvelle colonne 'Q'
df_sample['Q'] = df_sample['question'].apply(extract_second_q_and_r)

df_sample.head(1)

,context,question,answer,source_doc,Q
0,La marque française Secret's Lab complète sa c...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv,Quel est le ratio PG/VG du e-liquide The Drago...


In [34]:
df_sample['A'] = df_sample['question'].str.split(r'\nRéponse :').str[2]
df_sample.head(1)

,context,question,answer,source_doc,Q,A
0,La marque française Secret's Lab complète sa c...,\nVotre tâche est de rédiger une question fact...,\nVotre tâche est de rédiger une question fact...,../../RGBD/table_produits/produits.csv,Quel est le ratio PG/VG du e-liquide The Drago...,Le ratio PG/VG du e-liquide The Dragon de Sec...


In [35]:
df_sample.isna().sum()

context       0
question      0
answer        0
source_doc    0
Q             0
A             0
dtype: int64

In [36]:
# Garder seulement les colonnes 'context', 'Q' et 'A'
df_sample = df_sample[['context', 'Q', 'A']]
# Renommer les colonnes pour plus de clarté
df_sample = df_sample.rename(columns={ 'Q': 'question', 'A': 'answer'})
df_sample.head(1)

,context,question,answer
0,La marque française Secret's Lab complète sa c...,Quel est le ratio PG/VG du e-liquide The Drago...,Le ratio PG/VG du e-liquide The Dragon de Sec...


1.3. Setup critique agents - Vérification de la qualité des questions générées

Les questions générées par l’agent précédent peuvent comporter de nombreuses erreurs. Il est donc essentiel d’effectuer un contrôle qualité avant de les valider.

Pour cela, nous mettons en place des agents de critique qui évaluent chaque question selon plusieurs critères, définis dans cet article :

Fondement (Groundedness) : La question peut-elle être répondue en se basant uniquement sur le contexte fourni ?
Pertinence (Relevance) : La question est-elle pertinente pour les utilisateurs ? Par exemple, « Quelle est la date de sortie de Transformers 4.29.1 ? » n’est pas forcément utile pour des praticiens du Machine Learning.
Un autre problème fréquent concerne les questions qui dépendent fortement du contexte dans lequel elles ont été générées, mais qui sont incompréhensibles en dehors de celui-ci. Pour cela, nous introduisons un autre critère :

Autonomie (Stand-alone) : La question est-elle compréhensible sans contexte, pour une personne ayant des connaissances dans le domaine ou un accès à Internet ? À l’inverse, une question comme « Quelle est la fonction utilisée dans cet article ? », issue d’un blog spécifique, serait difficilement interprétable sans le contexte original.
Nous évaluons systématiquement chaque question selon ces critères. Si le score attribué par un de nos agents de critique est trop faible, la question est éliminée de notre ensemble de test.

💡 Méthodologie d’évaluation : Avant d’attribuer un score, nous demandons d’abord aux agents de fournir une justification de leur évaluation. Cela permet non seulement de vérifier les notes attribuées, mais aussi de donner au modèle plus d’informations pour réfléchir et élaborer sa réponse avant de la résumer sous forme d’un score unique.

In [40]:
import os
import openai
from langchain.chat_models import AzureChatOpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

azure_openai_api_key = os.getenv('AZURE_OPENAI_API_KEY_4')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_API_ENDPOINT_4')
deployment_name = os.getenv('AZURE_DEPLOYMENT_NAME_4')


# Set up Azure OpenAI API (for Azure deployment)
openai.api_type = "azure"
openai.api_key = azure_openai_api_key
openai.api_base = azure_openai_endpoint
openai.api_version = "2023-05-15"  # Ensure this is up to date

import warnings
warnings.filterwarnings('ignore')

In [41]:
llm = AzureChatOpenAI(
    api_key=azure_openai_api_key,  # Azure OpenAI API Key
    api_version="2023-12-01-preview",  # API version (you can adjust it as needed)
    azure_endpoint=azure_openai_endpoint,  # Azure OpenAI endpoint
    deployment_name=deployment_name,  # Ensure deployment_name is the correct one you configured in Azure
    temperature=0  # Optional: Adjust the temperature for randomness in responses
)

In [25]:
question_groundedness_critique_prompt = """
Vous allez recevoir un contexte et une question.
Votre tâche est de fournir une "note totale" évaluant dans quelle mesure la question peut être répondue de manière claire et sans ambiguïté avec le contexte fourni.
Évaluez votre réponse sur une échelle de 1 à 5, où 1 signifie que la question ne peut pas du tout être répondue avec le contexte donné, et 5 signifie que la question peut être clairement et sans ambiguïté répondue avec le contexte.

Fournissez votre réponse comme suit :

Réponse:::
Évaluation : (votre raisonnement pour la note, sous forme de texte)
Note totale : (votre note, un nombre entre 1 et 5)

Vous DEVEZ fournir des valeurs pour 'Évaluation :' et 'Note totale :' dans votre réponse.

Voici maintenant la question et le contexte.

Question : {question}\n
Contexte : {context}\n
Réponse:::  """

question_relevance_critique_prompt = """
Vous allez recevoir une question.
Votre tâche est de fournir une "note totale" représentant l'utilité de cette question pour les développeurs de machine learning travaillant sur des applications NLP avec l'écosystème Hugging Face.
Évaluez votre réponse sur une échelle de 1 à 5, où 1 signifie que la question n'est pas du tout utile, et 5 signifie que la question est extrêmement utile.

Fournissez votre réponse comme suit :

Réponse:::
Évaluation : (votre raisonnement pour la note, sous forme de texte)
Note totale : (votre note, un nombre entre 1 et 5)

Vous DEVEZ fournir des valeurs pour 'Évaluation :' et 'Note totale :' dans votre réponse.

Voici maintenant la question.

Question : {question}\n
Réponse::: """

question_standalone_critique_prompt = """
Vous allez recevoir une question.
Votre tâche est de fournir une "note totale" représentant dans quelle mesure cette question est indépendante du contexte.
Évaluez votre réponse sur une échelle de 1 à 5, où 1 signifie que la question dépend d'informations supplémentaires pour être comprise, et 5 signifie que la question a un sens par elle-même.
Par exemple, si la question fait référence à un contexte particulier, comme "dans le contexte" ou "dans le document", la note doit être de 1.
Les questions peuvent contenir des termes techniques obscurs ou des acronymes comme Gradio, Hub, Hugging Face ou Space et recevoir tout de même une note de 5 : il suffit que l'opérateur avec accès à la documentation puisse comprendre de quoi il s'agit.

Par exemple, la question "Quel est le taux de nicotine recommandé pour une cigarette électronique équipée d'une résistance inférieure à 1 ohm et d'une puissance supérieure à 30 watts ?" devrait recevoir une note de 1, car elle fait référence explicitement à un contexte précis (le type de cigarette électronique), et donc la question n'est pas indépendante du contexte.

Fournissez votre réponse comme suit :

Réponse:::
Évaluation : (votre raisonnement pour la note, sous forme de texte)
Note totale : (votre note, un nombre entre 1 et 5)

Vous DEVEZ fournir des valeurs pour 'Évaluation :' et 'Note totale :' dans votre réponse.

Voici maintenant la question.

Question : {question}\n
Réponse::: """

In [42]:
df_sample = df_sample[:20]

In [44]:
import json

def call_llm(llm, prompt: str):
    response = llm.invoke(prompt)  # Utilisation de invoke() au lieu de post()
    return response.content  # Récupération du texte généré


In [47]:
# Ajouter les critiques pour chaque QA
print("Generating critique for each QA couple...")
for output in tqdm(outputs):
    evaluations = {
        "groundedness": call_llm(
            llm,
            question_groundedness_critique_prompt.format(context=output["context"], question=output["question"]),
        ),
        "relevance": call_llm(
            llm,
            question_relevance_critique_prompt.format(question=output["question"]),
        ),
        "standalone": call_llm(
            llm,
            question_standalone_critique_prompt.format(question=output["question"]),
        ),
    }
    
    try:
        # Extraire les scores et évaluations
        for criterion, evaluation in evaluations.items():
            score, eval = (
                int(evaluation.split("Total rating: ")[-1].strip()),  # Score
                evaluation.split("Total rating: ")[-2].split("Evaluation: ")[1],  # Evaluation
            )
            output.update(
                {
                    f"{criterion}_score": score,
                    f"{criterion}_eval": eval,
                }
            )
    except Exception as e:
        print(f"Error evaluating {output['question']}: {e}")
        continue  # Passer à la génération suivante en cas d'erreur

# Créer un DataFrame avec les critiques
df_with_critique = pd.DataFrame(outputs)

Generating critique for each QA couple...


0it [00:00, ?it/s]


In [48]:
df_with_critique

""


In [ ]:
df_with_critique.to_csv('QA_test_samples_eval.csv', index=False)

In [1]:
import pandas as pd

In [6]:
df = pd.read_csv("../../app/recommendations.csv") 
df 

,Question,Response,Document Content
0,je veux liquide avec cassis,"Pour un e-liquide avec saveur de cassis, vous ...",index: 69\nurl: https://www.aromes-et-liquides...
1,je veux liquide avec fraise,Voici quelques options d'e-liquides avec saveu...,index: 99\nurl: https://www.aromes-et-liquides...
2,liquide avec lime,Il existe plusieurs e-liquides avec des saveur...,index: 28\nurl: https://www.aromes-et-liquides...
3,je veux pomme,Il semble que vous soyez intéressé par un prod...,index: 68\nurl: https://www.aromes-et-liquides...
4,je veux liquide avec fraise,Il existe plusieurs e-liquides avec saveur de ...,index: 99\nurl: https://www.aromes-et-liquides...
5,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...
6,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...
7,citron,"The term ""citron"" in the provided context refe...",index: 43\nurl: https://www.aromes-et-liquides...
8,citron,"The term ""citron"" in the provided context refe...",index: 43\nurl: https://www.aromes-et-liquides...
9,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...


In [1]:
import pandas as pd

# Définir les données à mettre dans le CSV
data = {
    "Timestamp": [],
    "Query": [],
    "Response": [],
    "Source": [],
    "Similarity Query-Response": [],
    "Similarity Response-Source": [],
    "Latency (ms)": []
}

# Créer un DataFrame à partir des données
df = pd.DataFrame(data)

# Enregistrer le DataFrame dans un fichier CSV
df.to_csv('monitoring.csv', index=False)

print("Le fichier monitoring.csv a été créé avec succès.")


Le fichier monitoring.csv a été créé avec succès.


In [10]:
df = pd.read_csv("../../app/recommendations.csv")
df

,Question,Response,Document Content
0,je veux liquide avec cassis,"Pour un e-liquide avec saveur de cassis, vous ...",index: 69\nurl: https://www.aromes-et-liquides...
1,je veux liquide avec fraise,Voici quelques options d'e-liquides avec saveu...,index: 99\nurl: https://www.aromes-et-liquides...
2,liquide avec lime,Il existe plusieurs e-liquides avec des saveur...,index: 28\nurl: https://www.aromes-et-liquides...
3,je veux pomme,Il semble que vous soyez intéressé par un prod...,index: 68\nurl: https://www.aromes-et-liquides...
4,je veux liquide avec fraise,Il existe plusieurs e-liquides avec saveur de ...,index: 99\nurl: https://www.aromes-et-liquides...
5,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...
6,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...
7,citron,"The term ""citron"" in the provided context refe...",index: 43\nurl: https://www.aromes-et-liquides...
8,citron,"The term ""citron"" in the provided context refe...",index: 43\nurl: https://www.aromes-et-liquides...
9,caramel,I don't know.,index: 46\nurl: https://www.aromes-et-liquides...
